## Agent AI - BACEN SGS Time-series analysis assistant

## NLP classification problem: Integretion with LLM Model (Meta-llama/Llama-3.2-3B-Instruct) 

Plug in a Large Language Model (LLM) to automatically interpret and explain the statistical outputs dynamically.

In [1]:
# fetch_bacen.py
import requests # access HTTP content
from requests.adapters import HTTPAdapter, Retry
import pandas as pd # lib for data manipulation

from tqdm import trange
import unicodedata # text normalization

import difflib
from difflib import SequenceMatcher
from bcb import sgs
import re

## sentence_transformers enbedding models
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
from openai import OpenAI
from dotenv import load_dotenv # load environment variables
import os


## Timne-series diagnostics tests
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import warnings
warnings.filterwarnings("ignore")

/home/max/projects/NLP/AI_Agent_Bacen/venv_ai_bacen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Testing OpenAI Connection

In [3]:
# 1. Initialize client

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello from my SageMaker instance!"}]
)

print(response.choices[0].message.content)

Hello from your SageMaker instance! If you have any questions or need assistance with anything, feel free to ask!


### Testing Transformers - Alucinates - to borader

In [13]:
from transformers import pipeline

# lightweight model (~1B parameters)
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

prompt = """
Explain in simple terms why a non-stationary time series needs differencing before applying ARIMA.
"""
response = pipe(prompt, max_new_tokens=150, do_sample=True)
print(response[0]["generated_text"])


Device set to use cpu



Explain in simple terms why a non-stationary time series needs differencing before applying ARIMA.

Differencing is used in statistical modeling to remove trends and seasonal patterns from time series data. It is a technique used to remove the effects of lagged values, which can affect the accuracy and interpretation of the predictions.

In the case of a non-stationary time series, when we apply ARIMA model on this time series, we may get a time series with stationary values, but with non-stationary trends. This can result in poor model performance, as the model may not be able to predict the future values of the time series accurately.

To address this problem, ARIMA is used to correct the time series by taking the difference of the time series before applying ARIMA model.


In [ ]:
summary_stats = {
    "mean": 0.0299,
    "std": 0.8877,
    "skewness": 0.105,
    "kurtosis": 2.1,
    "adf_p": 0.0
}

# Professional structured prompt
prompt = f"""
You are a data scientist specialized in econometrics and time-series modeling.

Below are summary diagnostics for a raw series:
- Mean: {summary_stats['mean']}
- Standard Deviation: {summary_stats['std']}
- Skewness: {summary_stats['skewness']}
- Kurtosis: {summary_stats['kurtosis']}
- ADF Test p-value: {summary_stats['adf_p']}

Tasks:
1. Assess whether the series appears stationary based on the ADF test.
2. Interpret what the mean and standard deviation indicate about level and volatility.
3. Comment on the distribution shape using skewness and kurtosis.
"""

# Generate explanation
response = pipe(prompt, max_new_tokens=200, do_sample=True, temperature=0.7)
print(response[0]["generated_text"])


You are a data scientist specialized in econometrics and time-series modeling.

Below are summary diagnostics for a raw series:
- Mean: 0.0299
- Standard Deviation: 0.8877
- Skewness: 0.105
- Kurtosis: 2.1
- ADF Test p-value: 0.0

Tasks:
1. Assess whether the series appears stationary based on the ADF test.
2. Interpret what the mean and standard deviation indicate about level and volatility.
3. Comment on the distribution shape using skewness and kurtosis.
4. Provide a concise executive-style summary that could be displayed on a Streamlit dashboard.

Interpretation of the Mean and Standard Deviation:

The mean is 0.0299 and the standard deviation is 0.8877. These values indicate that the series is not stationary.

The skewness of the series is 0.105, and the kurtosis is 2.1. These metrics indicate that the distribution is highly skewed to the right.

The ADF test shows that there is no evidence of a serial correlation, which means the series is stationary.

Tasks:
1. Evaluate the pre